[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Optimal_Transport.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Optimal Transport

The geometry of *moving probability mass*: Monge's dirt-shoveling problem, Kantorovich's relaxation, the 20-line Sinkhorn algorithm (verified against exact solvers), and why Wasserstein distances fixed a real failure mode of KL in machine learning.

## 1. Pre-requisites

[Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) (KL, for the contrast), [Convex Optimization II](../Intro_Math/Optimization/Convex_Optimization_2.ipynb) (duality — OT is a linear program).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment, linprog
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Monge, Kantorovich & Why KL Fails* (~35 min)
**Goal:** transport as the distance that respects GEOMETRY; where KL divergence goes blind.
**Feeds into:** Session 2 (Sinkhorn).

---

## 2. A Distance That Knows the Ground

💡 **Intuition.** KL divergence compares distributions *pointwise*: two non-overlapping distributions are 'infinitely different' whether they're 1 mm or 1 km apart — KL never looks at the ground between them. **Optimal transport** does: $W(p, q) = $ the minimum *cost of shoveling* $p$'s mass into $q$'s shape, cost = mass × distance moved. It's finite, smooth, and its gradient says *which direction to move* — exactly what a generative model's training signal needs (the insight behind Wasserstein GANs, and the geometry under [diffusion models](./Diffusion_Models.ipynb)).

In [ ]:
# two spikes sliding apart: KL slams to a wall instantly, W1 reports the distance
    # W1 in 1-D has a CLOSED FORM: integral |CDF_p − CDF_q|  (our oracle for later, too)

# YOUR CODE HERE


**What just happened.** Two curves telling opposite stories about the same pair of distributions. KL rises sharply and then **flattens** — once the spikes stop overlapping it has nothing further to say. $W_1$ rises as a **straight line with slope 1.000**, forever.

**That slope is not approximately 1; it is exactly what the theory demands.** Moving a unit of mass a distance $s$ costs $s$, so $W_1 = s$ and $dW_1/ds = 1$. The fit returns 1.000, which is a genuine verification rather than a plausible-looking plot — and it is available because in one dimension $W_1(p,q) = \int|F_p - F_q|$ has a closed form, computed here as `sum(|cumsum(p) - cumsum(q)|) * dx`. **Two lines of NumPy, no solver, exact answer.**

**Now read KL's flat region as the failure it is.** KL compares distributions **pointwise**: it evaluates $\log(p/q)$ where $p$ has mass and never asks how far away $q$'s mass sits. Once the supports separate, every additional metre of separation changes nothing — KL is blind to the ground between them. Two distributions 1 mm apart and 1 km apart get the same score.

**And a flat loss has zero gradient, which is the whole reason this matters in machine learning.** Train a generative model whose samples do not yet overlap the data. A KL-style objective reports "wrong" and provides **no direction** — the gradient has died and the generator cannot learn which way to move. This is exactly the failure that motivated Wasserstein GANs: replacing the divergence with $W_1$ restores a gradient that always points toward the data, because the cost keeps growing with distance.

**Note the small numerical dodge in the KL computation, since it is doing real work.** The mask `(p > 1e-12) & (q > 1e-12)` restricts the sum to the overlap; without it the true KL would be $+\infty$ the moment supports disjoin, and the plot would be a vertical line. So the visible saturation is a **floor imposed by finite precision on a Gaussian tail** — the honest version is worse than what is drawn. That strengthens the point rather than weakening it.

**One caveat about what generalises.** The exact slope and the closed form are **one-dimensional facts**. In higher dimensions $W_1$ is a linear program with no analytic solution, which is why the rest of this workshop exists — Session 2's Sinkhorn algorithm is what makes this distance computable at scale. The qualitative contrast (KL blind to geometry, $W$ aware of it) survives in any dimension; the two-line computation does not.

**Finally, resist concluding that KL is simply worse.** KL is the right object when densities overlap and likelihood is what you care about — it underlies maximum likelihood, variational inference, and the ELBO. Its indifference to geometry is a *feature* on a space with no meaningful distance (categorical labels) and a fatal flaw on one that has (images, point clouds, waveforms). **Match the divergence to whether your sample space has a metric that means something.**

---
### 🕐 Session 2 of 3 — *Sinkhorn: Entropic OT in 20 Lines* (~40 min)
**Goal:** add an entropy smoothing term and OT becomes two alternating normalizations — verified twice.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (OT in ML).

---

## 3. The Algorithm That Made OT Practical

💡 **Intuition.** Exact OT is a linear program — expensive at scale. Add an entropy term $\varepsilon H(\pi)$ and the optimal plan becomes $\pi = \mathrm{diag}(u) K \mathrm{diag}(v)$ with $K = e^{-C/\varepsilon}$ — and finding $u, v$ is just **alternately normalizing rows and columns** to match the marginals. Twenty lines, all matrix-vector, GPU-loving. As $\varepsilon \to 0$ it approaches the exact plan; we verify against *two* independent oracles: the Hungarian assignment solver and a general LP.

In [ ]:
# ORACLE 1: assignment case (uniform marginals, n=n) — Hungarian algorithm is exact
# ORACLE 2: general marginals — solve the LP directly

# YOUR CODE HERE


**What just happened.** A twenty-line loop of matrix–vector products matched two *independent* exact solvers:

| oracle | exact | Sinkhorn | gap |
|---|---|---|---|
| Hungarian (assignment) | 0.47865 | 0.48071 | **+2.06e−3** |
| LP via `linprog` | 0.06211 | 0.06145 | **−6.61e−4** |

**Two oracles rather than one is the point of this cell.** A method can agree with a single reference because both share an assumption or a bug. The Hungarian algorithm and a general-purpose LP solver share no code with each other or with Sinkhorn, and neither knows anything about entropic regularisation. **Agreement across three unrelated routes is evidence; agreement with yourself is not.**

**The first gap is positive, and it has to be.** The entropic plan is deliberately blurrier than the sharp optimal one, and any plan other than the LP optimum costs more. $+2\times10^{-3}$ on a cost of 0.48 is a 0.4% bias — exactly the price paid for a smooth, GPU-friendly problem, and it shrinks with $\varepsilon$.

**The second gap is negative, which is impossible — and worth stopping on.** Nothing can cost less than the optimum of the linear program. So the returned $\pi$ **is not a feasible coupling**: after 2000 finite iterations the row and column sums only approximately equal $a_2$ and $b_2$, and $\langle\pi, C\rangle$ evaluated on an infeasible plan is not a transport cost at all. Check it directly with `P.sum(1) - a2`. **A negative gap is a diagnostic that says "your marginals have not converged"**, not a discovery — and the habit of noticing when a number is on the wrong side of a bound is more valuable than either estimate.

**There is a second reason to be suspicious at $\varepsilon = 0.003$, and it is arithmetic.** With costs up to 1, the kernel $K = e^{-C/\varepsilon}$ spans from $e^0 = 1$ down to $e^{-333} \approx 10^{-145}$ — a hundred and forty-five orders of magnitude in one array. Double precision holds it, barely. Halve $\varepsilon$ again, or scale the costs up, and $K$ **silently underflows to zero**, the divisions produce `nan`, and the loop fails with no error. This is why every production implementation runs in **log-space** with log-sum-exp rather than forming $K$ directly.

**Note how little the algorithm is doing, because that is the achievement.** Two lines in the loop — `v = b/(Kᵀu)`, `u = a/(Kv)` — alternately rescale columns and rows until both marginals are satisfied. No pivoting, no factorisation, no combinatorial search. The optimal entropic plan has the form $\mathrm{diag}(u)K\mathrm{diag}(v)$, and finding $u, v$ is nothing but repeated normalisation. **That is why OT went from a $O(n^3\log n)$ linear program nobody could afford in a training loop to an $O(n^2)$ matrix operation that runs on a GPU.**

**Finally, be clear about what regularisation changed.** Sinkhorn does not approximate the LP with a heuristic — it solves a **different problem** (OT plus an entropy term) exactly, and that problem converges to the original as $\varepsilon \to 0$. The bias in the first row is not error; it is the answer to the question that was actually asked.

In [ ]:
# the transport plan itself, at three smoothing levels — sharpening toward the LP vertex

# YOUR CODE HERE


**What just happened.** The same transport problem at three smoothing levels, and the plan visibly **sharpens** as $\varepsilon$ falls. At $\varepsilon = 0.1$ the plan is a broad diffuse band — mass from each source is spread across many destinations. At $\varepsilon = 0.003$ it has collapsed toward a thin diagonal ridge, close to the sharp vertex the linear program returns. The reported cost falls monotonically alongside it.

**The blur is not numerical error; it is the entropy term doing exactly what it was added to do.** The regularised objective is $\langle\pi, C\rangle + \varepsilon H(\pi)$, and entropy is maximised by spreading mass out. So $\varepsilon$ literally buys diffusion: **larger $\varepsilon$ means the optimiser is paid to be uncertain**. As $\varepsilon \to 0$ that payment vanishes and the solution returns to the unregularised optimum.

**Which is why the sharpest picture is not simply the best one.** The trade-off runs in both directions:

| | large ε | small ε |
|---|---|---|
| plan | blurry | sharp |
| cost | biased high | accurate |
| convergence | fast | slow |
| arithmetic | stable | precarious |
| gradients | smooth, usable | near-discontinuous |

**The last row is why machine learning often *prefers* a blurry plan.** An LP optimum sits at a vertex, so it moves discontinuously as the inputs change — useless as a differentiable loss. Entropic smoothing makes the transport cost a smooth function of the marginals, which is precisely what lets Wasserstein-style losses be backpropagated through. **The regulariser was introduced for speed and turned out to be necessary for differentiability.**

**Note the structure that survives at every $\varepsilon$.** All three panels concentrate near the diagonal, because $C_2 = |x_i - y_j|$ makes nearby points cheap. Smoothing changes *how tightly* mass clusters around the cheap pairs, never *which* pairs are cheap. The geometry is in the cost matrix; $\varepsilon$ only controls how literally it is obeyed.

**One caution about pushing $\varepsilon$ lower than 0.003 in this code.** $K = e^{-C/\varepsilon}$ already spans 145 orders of magnitude here. Halve $\varepsilon$ again and entries underflow to exactly zero, the divisions produce `nan`, and the plot goes blank with no error raised — which is also why the $\varepsilon = 0.003$ cost in the previous cell landed *below* the LP optimum. **The sharpest column in this figure is at the edge of what naive float64 Sinkhorn can do**, and log-domain implementations exist for everything beyond it.

---
### 🕐 Session 3 of 3 — *OT in Machine Learning* (~35 min)
**Goal:** Wasserstein losses, barycenters, and domain adaptation — three working miniatures.
**Builds on:** Session 2.

---

## 4. Three Jobs for a Geometric Distance

In [ ]:
# (1) Wasserstein barycenter of three histograms — the *geometric* average
# vs the naive pointwise average (which invents mass in the middle)
# 1-D W-barycenter has a quantile closed form: average the inverse CDFs (another oracle-friendly fact)

# YOUR CODE HERE


**What just happened.** Three spikes at $x = 2$, $5$, $8$, averaged two ways. The pointwise mean is **three ghosts**, each at a third of the original height, sitting exactly where the inputs were. The Wasserstein barycenter is **one spike at $x = 5$**, the same shape and height as its inputs.

**The left panel is the failure, and it is worth naming precisely.** Averaging densities pointwise produces something that is not a member of the family you averaged. Three unimodal distributions average to a trimodal one; three faces average to a blur; three shapes average to a fog. **The arithmetic is correct and the answer is useless**, because "average" applied coordinate-wise is not the operation anyone meant.

**The right panel averages *positions* instead of *heights*, and in 1-D that has a closed form.** Average the **inverse CDFs** — the quantile functions — which is exactly what `np.interp(qs, cumsum(h), grid)` computes: for each quantile level, where does each distribution put that quantile, and what is the mean of those locations? The median of three spikes at 2, 5, 8 is a spike at 5. The shape survives because the operation moves mass rather than mixing it.

**State the principle once, since it transfers everywhere.** Pointwise averaging **mixes** distributions; Wasserstein averaging **interpolates** them. Mixing gives you "one of these, chosen at random"; interpolating gives you "the thing halfway between". Those are genuinely different objects, and almost every application wants the second — shape averaging, colour transfer, template estimation in medical imaging, and the interpolation frames in generative models all rely on it.

**Note that the quantile trick is another instance of this workshop's verification discipline.** The 1-D barycenter, like the 1-D $W_1$ in Session 1, has an exact closed form requiring no solver. That gives a reference for checking Sinkhorn-based barycenter code in higher dimensions, where no such shortcut exists — and where the general algorithm is an alternating scheme with one Sinkhorn solve per input distribution per iteration.

**One thing the picture slightly flatters, in the interest of honesty.** The three inputs are identical up to translation, so their quantile functions differ by a constant and the average is exactly a translate — the shape is preserved *perfectly*. Give the inputs different widths and the barycenter's width is the average of the widths, which is still sensible but no longer identical to any input. The demo is chosen to make the contrast maximally clean; the general behaviour is interpolation, not preservation.

In [ ]:
# (2) domain adaptation: transport source samples onto the target cloud, then classify

# YOUR CODE HERE


**What just happened.** A classifier trained on the source domain scores **49%** on the rotated, shifted target — indistinguishable from a coin flip. Train the same classifier on OT-mapped source points and it scores **73%**, with **no target labels used anywhere**.

**Start with why 49% is the expected disaster and not a bug.** The target domain is the source rotated by $\theta = 0.9$ rad and translated to $(2.5, 1.0)$. The source classifier learned the boundary $x_1 = 0$, which after that rotation is simply in the wrong place. Ask what more source data would buy: **nothing**. This is distribution shift, and the failure is in the alignment between train and test, not in the amount of training. No sample size fixes a boundary that is pointed the wrong way.

**The fix uses transport as a coordinate change.** Sinkhorn returns a soft coupling between the two clouds; row-normalising it gives each source point a weighted average of the target points it is coupled to — a **barycentric map** carrying source coordinates into the target frame. Train there, and the boundary lands in target coordinates. The only inputs were the two unlabelled point clouds and a distance.

**Now the honest reading, because 73% is not success.** The mapping recovers most of the gap from chance, but a quarter of the target is still misclassified. The reason is worth stating as a principle: **OT aligns distributions, not semantics.** The algorithm's objective is to overlay one cloud on the other as cheaply as possible; it has no notion that the two classes must stay on their own sides. At $\theta = 0.9$ the cheapest overlay partially swaps them, and the classifier inherits that confusion.

**Which is exactly what the research literature addresses, so the mediocre number is a signpost rather than an embarrassment.** Class-regularised OT adds a penalty for couplings that mix labels; labelled or semi-supervised OT uses a few target labels to anchor the map; joint distribution OT transports $(x, y)$ pairs rather than $x$ alone. All three exist because the plain version does what this cell shows — better than nothing, short of solved.

**Two sample-size caveats before anyone quotes the number.** The target has 150 points, so a binomial standard error near 3.6%: the 24-point improvement is roughly seven standard errors and therefore real, but the value 73% is not reproducible to the digit, and one seed is a demonstration rather than a benchmark. And `acc(w_naive) = 49%` is a coin flip *in expectation*; a different seed could put it at 55% and change the apparent size of the win.

**Finally, note what makes this attractive despite the imperfection.** Nothing in the pipeline required labels from the target domain — the expensive resource in every real adaptation problem. A method that converts chance into 73% using only unlabelled data and a distance is worth having, provided you report it as what it is.

**(3) And the one you've already met:** [diffusion models](./Diffusion_Models.ipynb) learn a path between noise and data; the probability-flow view of that path is a transport map, and 'flow matching' — the current frontier — trains it with explicitly OT-inspired straight-line couplings. The shoveling metaphor became the state of the art.

## 5. Conclusion

OT measures distribution distance *through the ground metric* (where KL is blind), Sinkhorn computes it at scale (verified against Hungarian and LP oracles to 1e-3), and barycenters/adaptation/flows cash it in ML. Geometry, not pointwise comparison.

---
## Where next

- [Diffusion Models](./Diffusion_Models.ipynb) & the score/flow frontier.
- [Convex Optimization II](../Intro_Math/Optimization/Convex_Optimization_2.ipynb) — OT's duality (Kantorovich–Rubinstein) is a beautiful exercise.